# Harvest Now, Decrypt Later (Shor + PQC)

*Secure the Grid, step 2 · UTC Problem 4*

This problem is mostly analysis. The code here is a small proof of concept: factor 15 with Shor's algorithm. Parts 3 and 4 (exposure map and migration plan) are written — do them in your write-up.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

### Part 1 — Shor demo (factor 15)

The circuit below performs quantum period-finding for N = 15 with base a = 7 (a standard textbook case). Run it, read the measured period, and use it to recover the factors. The classical post-processing is left as a short `TODO`.

> **Before this goes on a slide:** look at `c_amod15_a7` in the next cell. Would it work for N = 21, or for a 2048-bit number? What does that say about what this demo proves, and what it doesn't?


In [ ]:
def c_amod15_a7(power):
    """Controlled multiply by 7^power mod 15, hardcoded for this demo.

    Shor uses it to find the period of 7^x mod 15. A real RSA number needs a general multiply here.
    """
    U = QuantumCircuit(4)
    # One pass multiplies by 7 mod 15; repeat it `power` times.
    for _ in range(power):
        U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        for q in range(4):
            U.x(q)
    U = U.to_gate(); U.name = f"7^{power} mod 15"
    # The control qubit turns that multiply on or off.
    return U.control()

def qft_dagger(n):
    """Inverse Fourier transform on the counting qubits.

    It turns the phase from the modular multiply into a bitstring we can measure.
    """
    qc = QuantumCircuit(n)
    # Swap the ends so the rotations below are the inverse QFT.
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    # Negative controlled phases, then a Hadamard, on each qubit.
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc


In [ ]:
n_count = 8                      # counting qubits
qc = QuantumCircuit(n_count + 4, n_count)
# Every exponent x at once.
qc.h(range(n_count))
qc.x(n_count)                    # set work register to |1>
# Each counting qubit controls multiply-by-7^(2^i) mod 15.
for i in range(n_count):
    qc.append(c_amod15_a7(2 ** i), [i] + list(range(n_count, n_count + 4)))
# Turn the phase into a bitstring and measure it.
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
measured = max(counts, key=counts.get)
phase = int(measured, 2) / (2 ** n_count)
print("Measured phase:", phase)


In [ ]:
from fractions import Fraction
from math import gcd

# Each run lands on one of four phases (0, 1/4, 1/2, 3/4) about equally often.
# Phases 0 and 1/2 give a wrong period, so try the results from most to least common
# and keep the first r that is even and passes the check 7^r mod 15 = 1.
ranked = sorted(counts, key=counts.get, reverse=True)
for measured in ranked:
    phase = int(measured, 2) / (2 ** n_count)
    # Denominator of the phase is the candidate period.
    r = Fraction(phase).limit_denominator(15).denominator
    print(f"measured {measured} -> phase {phase} -> candidate period r = {r}, 7^r mod 15 = {pow(7, r, 15)}")
    if r % 2 == 0 and pow(7, r, 15) == 1:
        break

# Factors of 15 from the period that passed the check.
p = gcd(7 ** (r // 2) - 1, 15)
q = gcd(7 ** (r // 2) + 1, 15)
print(f"factors of 15: {p} and {q}")

### Part 2 — how far is real hardware?

List the real devices on qBraid and record their qubit counts. Then estimate how many qubits breaking RSA-2048 would need, and state the gap.


In [ ]:
# List qBraid's gate-based quantum computers and their qubit counts
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
analog = ("quera:qpu:aquila", "pasqal:qpu:fresnel")      # analog machines: they can't run Shor's circuits
for d in provider.get_devices():
    if ":qpu:" in d.id and not d.id.endswith(analog):
        print(f"{d.id:42s} {d.num_qubits} qubits")
# The same machine can appear under several routes (direct, aws, azure, openquantum): count it once.

# Then, in your write-up:
#  - largest real device qubit count you found: ______
#  - qubits needed to break RSA-2048: ______ logical, ______ physical
#    Cite the source, year and assumptions: the estimate has fallen a lot. Start with Gidney & Ekerå
#    (2019) and Gidney (2025, arXiv:2505.15917), then look for anything newer.
#  - the gap: ______ orders of magnitude in qubit count. What does a qubit count leave out?

## What to have ready

- **Part 1:** factors of 15 from the measured period (above).
- **Part 2:** device qubit counts and gap notes (above).
- **Parts 3–4:** exposure table and migration order (above); full assumptions in `data/systems.json`.
- **Part 5:** next, `2b_grover_symmetric_keys.ipynb`.


## Part 2 notes — hardware gap

From qBraid’s live gate-based device list (analog machines left out), the largest machines we saw were on the order of ~10² qubits (for example Rigetti around 107). Published estimates for breaking RSA-2048 have fallen a lot: Gidney & Ekerå (2019) talked about ~20 million noisy qubits; Gidney (2025, arXiv:2505.15917) is under ~1 million. Even with the newer number we are still many orders of magnitude short today — but the *recording* of encrypted traffic can start now (“harvest now, decrypt later”).

## Parts 3–4 — exposure map and migration (our risk model)

We listed the utility systems that still lean on RSA/ECC (SCADA, ICCP, VPN, PKI, firmware signing, GOOSE tags, AMI, time sync, and so on) in `data/systems.json`. We score each one with a two-clock model (Mosca’s rule, Monte Carlo):

- **Secrecy clock:** can traffic recorded today be read once RSA/ECC breaks?
- **Authenticity clock:** can someone forge certificates or firmware after Q-day before we finish migrating?

Impact is weighted by NERC CIP (High / Medium / Low). Gaps that need no quantum computer (no auth, no replay protection, no firmware rollback) are listed separately so they do not blur the quantum score.

In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
for _cand in (_here, _here.parent):
    if (_cand / "secure_grid").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

import numpy as np
import pandas as pd
from secure_grid import risk

data = risk.load()
table_2025, z_2025 = risk.run(data, estimate="2025")
table_2019, z_2019 = risk.run(data, estimate="2019")
THIS_YEAR = risk.THIS_YEAR

print("Quantum arrival (2025 estimate): median",
      int(round(np.median(THIS_YEAR + z_2025))),
      "| 1 in 10 chance by", int(round(np.percentile(THIS_YEAR + z_2025, 10))))
print("Quantum arrival (2019 estimate): median",
      int(round(np.median(THIS_YEAR + z_2019))),
      "| 1 in 10 chance by", int(round(np.percentile(THIS_YEAR + z_2019, 10))))
print()
cols = ["system", "impact", "P(exposed)", "risk score",
        "exposed today, no quantum needed", "must be fixed before"]
display_df = table_2025[cols].copy()
display_df["P(exposed)"] = display_df["P(exposed)"].map(lambda x: f"{x:.0%}")
display_df["risk score"] = display_df["risk score"].map(lambda x: f"{x:.2f}")
print(display_df.to_string(index=False))

## Migration plan (what we would tell the utility)

1. **Close today’s gaps first** (firmware rollback, unprotected time, plaintext links, MFA on remote access) — no quantum computer required for those attacks.
2. **Move the PKI / root of trust** to post-quantum certificates next. SCADA, ICCP, VPN, and GOOSE all wait on it.
3. **Then by risk score:** ICCP and SCADA, BCSI/firmware, AMI, VPN, GOOSE, with ML-KEM for keys and ML-DSA / hash-based signatures (SP 800-208) for firmware.
4. **NIST IR 8547:** deprecate RSA/ECC by 2030, disallow by 2035 — our median “years to fix” for several systems still sits past that if we wait, so start now.

One research paper (2019 → 2025 qubit estimate) moved our median arrival several years closer. Doubling time and how long secrets must stay secret dominate the ranking.